# Evaluating RAG Pipeline Output Quality

This notebook is the runnable companion to [Evaluating Pipeline Output Quality](https://qdrant.tech/documentation/search-evaluation/pipeline-output-quality/). It builds a small RAG pipeline over the BioASQ mini benchmark, measures retrieval with recall@10, and scores the generated answers with Ragas. It then changes one thing, the generation prompt, to show how pairing a retrieval score with an answer score tells you which half of the pipeline moved.

You need one [OpenRouter](https://openrouter.ai/) API key. In Colab, add it under **Secrets** as `OPENROUTER_API_KEY`. Retrieval embeddings run locally with FastEmbed, and Qdrant runs in local mode, so there is nothing else to set up. A full run at N=100 cost about $1.23 in OpenRouter credits on October 8, 2026.

In [1]:
%pip install -q "ragas==0.4.3" "langchain-community==0.4.1" "qdrant-client[fastembed]==1.19.1" "openai==3.3.0" datasets pandas

Note: you may need to restart the kernel to use updated packages.


`langchain-community` is pinned because version 0.4.2 removed a module that Ragas 0.4.3 imports, and `import ragas` fails with it.

In [2]:
import os
from getpass import getpass

try:
    from google.colab import userdata

    os.environ.setdefault("OPENROUTER_API_KEY", userdata.get("OPENROUTER_API_KEY"))
except Exception:
    pass  # not in Colab, or the secret is not set
if not os.environ.get("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter API key: ")

GENERATOR_MODEL = "anthropic/claude-haiku-5.5"
JUDGE_MODEL = "openai/gpt-5.4-nano"  # a different model family from the generator
JUDGE_EMBEDDING_MODEL = "openai/text-embedding-3-small"
RETRIEVAL_MODEL = "BAAI/bge-small-en-v1.5"

N = 100  # questions to evaluate; more questions give more stable means
SEED = 42
K = 10  # passages retrieved per question
MAX_CONCURRENCY = 16

RECALL_TARGET = 0.5
FAITHFULNESS_TARGET = 0.8

## Load the Benchmark

[rag-mini-bioasq](https://huggingface.co/datasets/rag-datasets/rag-mini-bioasq) ships the three things an end-to-end evaluation needs: a passage corpus, questions labeled with their relevant passage IDs, and a reference answer for each question. The labels drive recall@10, and the reference answers drive `context_precision`.

About 30% of the corpus rows hold the literal string `nan` instead of text. The next cell drops them, drops the labels that point at them, and drops questions left with no label.

In [3]:
import json
import random

from datasets import load_dataset

corpus = load_dataset("rag-datasets/rag-mini-bioasq", "text-corpus", split="passages")
qa = load_dataset("rag-datasets/rag-mini-bioasq", "question-answer-passages", split="test")

passages = {row["id"]: row["passage"] for row in corpus if row["passage"].strip() != "nan"}

questions = []
dropped_labels = 0
for row in qa:
    labels = json.loads(row["relevant_passage_ids"])
    usable = [pid for pid in labels if pid in passages]
    dropped_labels += len(labels) - len(usable)
    if usable:
        questions.append(
            {"id": row["id"], "question": row["question"], "reference": row["answer"], "relevant": set(usable)}
        )

print(f"{len(passages):,} passages kept, {len(corpus) - len(passages):,} empty rows dropped")
print(f"{len(questions):,} questions kept, {len(qa) - len(questions):,} dropped, {dropped_labels:,} labels pointed at empty rows")

random.seed(SEED)
sample = random.sample(questions, N)

28,001 passages kept, 12,220 empty rows dropped
4,387 questions kept, 332 dropped, 12,925 labels pointed at empty rows


## Index the Corpus in Qdrant

Qdrant's built-in FastEmbed integration embeds each `models.Document` locally during upload, so there is no separate embedding step. Embedding the 28,001 passages on CPU took about 20 minutes on a laptop and can take longer on a free Colab runtime. Qdrant also prints a warning that local mode is meant for small collections. That is fine for this evaluation; run production collections on a Qdrant server or in Qdrant Cloud.

In [4]:
from qdrant_client import QdrantClient, models

client = QdrantClient(":memory:")
client.create_collection(
    "bioasq",
    vectors_config=models.VectorParams(
        size=client.get_embedding_size(RETRIEVAL_MODEL), distance=models.Distance.COSINE
    ),
)
client.upload_collection(
    "bioasq",
    ids=list(passages),
    vectors=[models.Document(text=text, model=RETRIEVAL_MODEL) for text in passages.values()],
    payload=[{"text": text} for text in passages.values()],
)
print(client.count("bioasq").count, "points")

28001 points


## Measure Retrieval

Recall@10 is the share of a question's relevant passages that appear in its top 10 results. It is the retrieval half of the diagnosis. A question with more than 10 labeled passages cannot reach 1.0, and BioASQ labels many passages for some questions, so read a low recall on those questions with care.

In [5]:
def recall_at_k(retrieved_ids, relevant_ids):
    return len(set(retrieved_ids) & relevant_ids) / len(relevant_ids)


assert recall_at_k([1, 2, 3], {1, 9}) == 0.5

for q in sample:
    points = client.query_points(
        "bioasq",
        query=models.Document(text=q["question"], model=RETRIEVAL_MODEL),
        limit=K,
        with_payload=True,
    ).points
    q["retrieved_contexts"] = [p.payload["text"] for p in points]
    q["recall"] = recall_at_k([p.id for p in points], q["relevant"])

print(f"mean recall@{K}: {sum(q['recall'] for q in sample) / N:.3f}")

mean recall@10: 0.668


## Generate Answers

The generator sees the retrieved passages through a grounding prompt. Keep the prompt in a versioned string so a prompt change is a visible, reviewable diff. The generator is capped at 1024 tokens per answer (`max_tokens=1024`). The judge scores the text as written, so raise the cap if your answers run long.

In [6]:
import asyncio

from openai import AsyncOpenAI

openrouter = AsyncOpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
    max_retries=5,
    timeout=60,
)
limit = asyncio.Semaphore(MAX_CONCURRENCY)

GROUNDED_PROMPT = """You are answering questions using retrieved source material.

Answer the question below using only the provided context.
If the context does not contain the answer, say so explicitly.
Do not rely on outside knowledge.

Context:
{retrieved_context}

Question:
{query_text}
"""


async def generate(prompt_template, q):
    prompt = prompt_template.format(
        retrieved_context="\n\n".join(q["retrieved_contexts"]), query_text=q["question"]
    )
    async with limit:
        try:
            response = await openrouter.chat.completions.create(
                model=GENERATOR_MODEL,
                max_tokens=1024,
                extra_body={"reasoning": {"enabled": False}},
                messages=[{"role": "user", "content": prompt}],
            )
        except Exception as error:
            print(f"generation failed: {type(error).__name__}: {error}")
            return ""  # an empty answer is scored as a pipeline failure
    return response.choices[0].message.content or ""


baseline_answers = await asyncio.gather(*(generate(GROUNDED_PROMPT, q) for q in sample))
print(baseline_answers[0])

Based on the provided context, the treatments of choice for GIST are:

- **Localized/primary disease:** Surgical resection is the treatment of choice. Complete surgical resection with negative margins (R0) is described as the best therapeutic option and the most definitive treatment. One source specifically states that laparoscopic technique is feasible and recommended as the treatment of choice for all patients with primary and localized GIST.

- **Locally advanced or metastatic/unresectable disease:** Imatinib mesylate has emerged as the initial treatment of choice, given either as cytoreductive or definitive treatment. Surgery or ablative modalities may be used in this setting, particularly when disease can be grossly resected or destroyed, or to manage complications after imatinib failure.

- **Progression after imatinib:** Several targeted therapies improve survival, including sunitinib and regorafenib, and more recently ripretinib (approved by the FDA for advanced GIST in adults 

## Score the Answers with Ragas

Three metrics cover the common failure modes:

- `faithfulness`: are the answer's claims supported by the retrieved passages?
- `answer_relevancy`: does the answer address the question?
- `context_precision`: are the passages that support the reference answer ranked near the top? It needs a reference answer, so it runs only on questions that have one.

The judge is a different model family from the generator, because a model judging its own output tends to score it higher. Each metric receives only the fields its `ascore()` accepts, and a failed judge call becomes a NaN that is reported instead of stopping the run.

In [7]:
import pandas as pd
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerRelevancy, ContextPrecision, Faithfulness

judge = llm_factory(
    JUDGE_MODEL,
    client=openrouter,
    max_tokens=4096,
    extra_body={"reasoning": {"enabled": False}},  # no thinking mode
)
judge_embeddings = embedding_factory("openai", model=JUDGE_EMBEDDING_MODEL, client=openrouter)

faithfulness = Faithfulness(llm=judge)
answer_relevancy = AnswerRelevancy(llm=judge, embeddings=judge_embeddings)
context_precision = ContextPrecision(llm=judge)


async def score(metric, **fields):
    async with limit:
        try:
            return (await metric.ascore(**fields)).value
        except Exception as error:
            print(f"{metric.name} failed: {type(error).__name__}: {error}")
            return float("nan")


async def score_answers(answers):
    faithfulness_scores = asyncio.gather(*(
        score(faithfulness, user_input=q["question"], response=a, retrieved_contexts=q["retrieved_contexts"])
        for q, a in zip(sample, answers)
    ))
    relevancy_scores = asyncio.gather(*(
        score(answer_relevancy, user_input=q["question"], response=a)
        for q, a in zip(sample, answers)
    ))
    return await faithfulness_scores, await relevancy_scores


async def score_context_precision(q):
    if not (q.get("reference") or "").strip():
        return float("nan")  # reference-based metric: nothing to score against
    return await score(
        context_precision, user_input=q["question"], reference=q["reference"], retrieved_contexts=q["retrieved_contexts"]
    )


baseline_faithfulness, baseline_relevancy = await score_answers(baseline_answers)
baseline = pd.DataFrame({
    "question": [q["question"] for q in sample],
    "recall@10": [q["recall"] for q in sample],
    "context_precision": await asyncio.gather(*(score_context_precision(q) for q in sample)),
    "faithfulness": baseline_faithfulness,
    "answer_relevancy": baseline_relevancy,
})
baseline.mean(numeric_only=True).round(3)

faithfulness failed: ValueError: response is missing. Please add response to the test sample.


answer_relevancy failed: ValueError: response cannot be empty


API call failed on attempt 1: Request timed out.


Max retries exceeded. Total attempts: 1, Last error: Request timed out.


context_precision failed: InstructorRetryException: Request timed out.


recall@10            0.668
context_precision    0.626
faithfulness         0.940
answer_relevancy     0.698
dtype: float64

In [8]:
baseline.isna().sum()  # judge calls that failed, per metric

question             0
recall@10            0
context_precision    1
faithfulness         1
answer_relevancy     1
dtype: int64

Averages hide the queries that fail. Start with the lowest faithfulness scores and read those answers next to their passages. Row `i` of the table matches `baseline_answers[i]` and `sample[i]["retrieved_contexts"]`.

In [9]:
baseline.nsmallest(10, "faithfulness")

,question,recall@10,context_precision,faithfulness,answer_relevancy
2,Which method for subsampling of NGS reads requ...,0.250000,0.250000,0.333333,0.000000
35,Which tool is used for the identification of r...,1.000000,0.166667,0.625000,0.577554
41,What is CellMaps?,0.000000,0.000000,0.666667,0.000000
53,Are PDXK mutations linked to polyneuropathy?,0.000000,0.142857,0.666667,0.897120
69,Which member of the human mycobiota is associa...,1.000000,1.000000,0.700000,0.783388
38,Which diseases are associated with Alu element...,0.333333,0.000000,0.733333,0.943977
16,What is the advantage of neutral loss detectio...,0.750000,0.566667,0.750000,0.942691
83,What is the disorder in which mutations in U4a...,1.000000,0.500000,0.750000,0.365039
3,What is the enzymatic activity of PARL?,1.000000,0.000000,0.777778,0.743572
34,List phagosomal markers.,0.500000,0.375000,0.777778,0.688247


## Change One Thing: The Prompt

The second run lets the generator use its own knowledge. Retrieval is untouched, so recall@10 and `context_precision` carry over unchanged, and any movement in the answer metrics belongs to generation.

In [10]:
OPEN_BOOK_PROMPT = """Answer the question below. The context may help, and you may also use your own knowledge.

Context:
{retrieved_context}

Question:
{query_text}
"""

changed_answers = await asyncio.gather(*(generate(OPEN_BOOK_PROMPT, q) for q in sample))
changed_faithfulness, changed_relevancy = await score_answers(changed_answers)
changed = baseline.assign(faithfulness=changed_faithfulness, answer_relevancy=changed_relevancy)

pd.DataFrame({
    "grounded prompt": baseline.mean(numeric_only=True),
    "open-book prompt": changed.mean(numeric_only=True),
}).round(3)

faithfulness failed: ValueError: response is missing. Please add response to the test sample.


answer_relevancy failed: ValueError: response cannot be empty


API call failed on attempt 1: Request timed out.


Max retries exceeded. Total attempts: 1, Last error: Request timed out.


answer_relevancy failed: InstructorRetryException: Request timed out.


,grounded prompt,open-book prompt
recall@10,0.668,0.668
context_precision,0.626,0.626
faithfulness,0.940,0.830
answer_relevancy,0.698,0.759


## Place Each Query in the 2x2

Pairing recall@10 with faithfulness for every query shows which half of the pipeline to investigate. The targets are set in the configuration cell; set them for your own system before you compare runs.

In [11]:
def quadrant(recall, faithfulness):
    retrieval = "high" if recall >= RECALL_TARGET else "low"
    generation = "high" if faithfulness >= FAITHFULNESS_TARGET else "low"
    return f"{retrieval} recall, {generation} faithfulness"


pd.DataFrame({
    name: run.dropna(subset=["faithfulness"])
    .apply(lambda row: quadrant(row["recall@10"], row["faithfulness"]), axis=1)
    .value_counts()
    for name, run in [("grounded prompt", baseline), ("open-book prompt", changed)]
}).fillna(0).astype(int)

,grounded prompt,open-book prompt
"high recall, high faithfulness",67,55
"high recall, low faithfulness",6,18
"low recall, high faithfulness",21,10
"low recall, low faithfulness",5,16


In this run, 67 of the 99 scored grounded answers land in high recall, high faithfulness. 13 of the 100 questions have more than 10 labeled passages, so they cannot reach a recall@10 of 1.0 even when every retrieved passage is relevant. The signal to read is the movement between the two columns. Recall@10 is identical in both runs, so every query that leaves high faithfulness under the open-book prompt moved because of generation.

## Record the Run

Both the generator and the judge sample their output, so a rerun changes the numbers. Keep the versions and model IDs next to every result you compare.

In [12]:
import datetime
import importlib.metadata

print("run date:", datetime.date.today())
for package in ["ragas", "langchain-community", "qdrant-client", "fastembed", "openai", "datasets", "pandas"]:
    print(f"{package}=={importlib.metadata.version(package)}")
print(f"generator={GENERATOR_MODEL} judge={JUDGE_MODEL} judge_embeddings={JUDGE_EMBEDDING_MODEL}")
print(f"retrieval={RETRIEVAL_MODEL} N={N} SEED={SEED} K={K}")

run date: 2026-10-08
ragas==0.4.3
langchain-community==0.4.1
qdrant-client==1.19.1
fastembed==0.8.1
openai==3.3.0
datasets==5.1.0
pandas==3.0.6
generator=anthropic/claude-haiku-5.5 judge=openai/gpt-5.4-nano judge_embeddings=openai/text-embedding-3-small
retrieval=BAAI/bge-small-en-v1.5 N=100 SEED=42 K=10
